# B2-019-attention-transformers — Practice p19 — Solution

**Type:** integrative · **Difficulty:** advanced · **Concepts:** transformer-residual-layernorm, position-wise-feed-forward, transformer-block

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb).

## Independent solution

The given recurrence mixes pre- and post-norm order, sends FFN the composite x+MHA(LN(x)) without naming its residual source, and wraps both branches in one final LayerNorm. The two explicit pre-norm recurrences are y=x+MHA(LN1(x)) and z=y+FFN(LN2(y)). Every intermediate has shape (B,N,D). Gradients have an identity residual route from z to y and from y to x in addition to the learned-sublayer routes. If attention is zero, y=x; if FFN is zero, z=y.

In [ ]:
import torch
from torch import nn
SEED = 20260808
ATOL = 1e-12
RTOL = 1e-12
torch.manual_seed(SEED)
B, N, D = 2, 3, 4
x = torch.arange(B * N * D, dtype=torch.float64).reshape(B, N, D) / 10.0
x.requires_grad_()
norm1 = nn.LayerNorm(D, dtype=torch.float64)
norm2 = nn.LayerNorm(D, dtype=torch.float64)
attention = nn.Linear(D, D, bias=False, dtype=torch.float64)
ffn = nn.Sequential(nn.Linear(D, 8, dtype=torch.float64), nn.ReLU(), nn.Linear(8, D, dtype=torch.float64))
with torch.no_grad():
    attention.weight.zero_()
    ffn[0].weight.zero_()
    ffn[0].bias.zero_()
    ffn[2].weight.zero_()
    ffn[2].bias.zero_()
y = x + attention(norm1(x))
z = y + ffn(norm2(y))
z.sum().backward()
gradient = x.grad.clone()

### Answer check

In [ ]:
assert y.shape == z.shape == (B, N, D)
assert torch.allclose(y, x.detach(), atol=ATOL, rtol=RTOL)
assert torch.allclose(z, y, atol=ATOL, rtol=RTOL)
assert gradient.shape == x.shape
assert torch.allclose(gradient, torch.ones_like(x), atol=ATOL, rtol=RTOL)